# 16.5 Padding 花掉多少工作？


訓練設定每段固定四格，兩份短文件各只有兩個token，若分別補到四格，就有八個位置、只有四個是真實內容。補進去的PAD雖然不算答案誤差，普通矩陣運算仍可能處理它們。把短文件接到同一段、充分使用這四格，叫sequence packing（序列裝填）。但我們若想讓兩份文件各自獨立，接起來時還要阻止它們互相讀取，不能只省空間而改了學習問題。

前置是[因果遮罩](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html)和[填充位置](https://birdhackor.github.io/tiny-perceptron-vlm/7.7.html)。PAD是補齊長度的符號，EOS是文件或回答結束的符號。EOS只是模型看到的一個token，不會讓注意力自動失明。例如A占位置0、1，B占2、3；普通因果遮罩允許位置3讀0至3，其中就包含A。若需要文件隔離，除了不能看未來，還要限制Query與Key屬於同一文件。

圖的列是Query所在位置，欄從左至右是可讀位置0至3。綠格代表允許，灰格代表禁止。B的兩列雖然排在A之後，左側兩欄仍為灰色；右上角禁止是因果限制，左下角禁止則來自跨文件隔離，兩者理由不同。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/packing.svg")))

In [ ]:
import torch
from tiny_perceptron.attention import attention_mask

position = torch.arange(4)
segment = torch.tensor([[0, 0, 1, 1]])
allowed = attention_mask(position, position, segments=segment)
print("隔離文件", allowed[0, 0].int().tolist())
causal_only = attention_mask(position, position)
print("只用因果的最後一列", causal_only[0, 0, -1].int().tolist())

`segment`每項記錄該位置的文件編號；0與1不是權重或時間，是兩份文件的身份。`attention_mask`把「Key位置不晚於Query」與「文件編號相同」取交集，返回batch、head、Query、Key四軸布林表。batch是同時處理的資料筆數，head是[一種注意力讀法](https://birdhackor.github.io/tiny-perceptron-vlm/3.7.html)；本例這兩軸的長度都為1，完整形狀是 `(1,1,4,4)`。`[0,0]`取第一段、第一頭，`int`把True/False印成1/0。四列應為 `[1,0,0,0]`、`[1,1,0,0]`、`[0,0,1,0]`、`[0,0,1,1]`，正好對應圖。只用因果的最後一列則為 `[1,1,1,1]`。

packing還需處理每份文件的位置編號，以及跨邊界的下一字答案。例如A的EOS後面雖接著B第一字，不應把「預測B第一字」當成A的目標，除非你有意訓練跨文件續寫。每份文件可以重新從位置0開始，但注意力遮罩仍要知道文內先後與segment，不能把兩個位置0誤當同一token。這段程式只核對隔離遮罩，沒有悄悄完成資料裝填的所有步驟。

練習只移除 `segments=segment`，先預測位置2也能讀0、1，與圖左下角不同，再執行核對。裝填減少PAD不代表同樣比例的速度收益：某些實作仍建立完整方陣，需看實際運算；若刻意允許文件互相讀取，也要將它記成另一種訓練設定。
